# Matrice-256 (FaNet v4.2) : Suite scientifique et étude d'ablation v5.0 (Kaggle GPU)

Ce notebook exécute sur GPU Kaggle l'ensemble des tests unitaires et des 8 expériences scientifiques corrigées de **Matrice-256 / bokono v4.2** (hypercube binaire 4 bits, requêtes cross-features, bloc résiduel FaReLU -> FaSigma, mini-batching GPU accéléré) :
1. Vérification du GPU (T4 / P100) et clonage du dépôt GitHub `CedBag/matrice-256`.
2. Installation des dépendances optionnelles (`onnx`, `onnxruntime`, `codecarbon`).
3. Exécution des tests unitaires (`pytest`) pour valider les fondations mathématiques.
4. Lancement de la suite complète `tests/run_benchmark_v5.py` (5-Fold x 5 seeds, 9 variantes contrôlées, tests de Wilcoxon, OOD sans fuite, ECE pondérée, MIA réel, ONNX).
5. Affichage des tableaux récapitulatifs et des figures générées.
6. Export de l'archive `/kaggle/working/results_v42.zip` prête à télécharger.

**Réglages Kaggle requis :** Accelerator = `GPU T4 x2` (ou `GPU P100`), Internet = `On`.

In [ ]:
import torch, platform
print("Python  :", platform.python_version())
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

### Analyse de l'environnement
Si `CUDA : True` apparaît ci-dessus, tous les entraînements PyTorch (`TorchFaNet`, `IsoMLP`, `IsoLSTM`) utiliseront automatiquement le GPU Kaggle.

In [ ]:
!rm -rf /kaggle/working/matrice-256
!git clone https://github.com/CedBag/matrice-256.git /kaggle/working/matrice-256
%cd /kaggle/working/matrice-256
!git log -n 3 --oneline
%pip install -q onnx onnxruntime codecarbon

### Vérification des tests unitaires (`bokono` v4.1)
Avant tout benchmark, on valide la matrice 16x16, la continuité de `FaReLU` et `TorchLearnableFaSigma`, la propagation des gradients et la sauvegarde/rechargement.

In [ ]:
!pytest tests/test_week1.py tests/test_v03.py tests/test_v30.py -v

### Interprétation des tests unitaires
Tous les tests doivent passer au vert (`PASSED`). Si un test échoue, ne pas lancer le benchmark complet avant d'en identifier la cause.

In [ ]:
!python tests/run_benchmark_v5.py --device auto --out-dir /kaggle/working/results_v42 --zip-path /kaggle/working/results_v42.zip

### Lecture du rapport complet et visualisation des graphiques réels
Les tableaux ci-dessous proviennent directement des exécutions multi-seeds et toutes les courbes (ROC MIA, fiabilité clinique, bruit, ablation, efficacité) sont tracées à partir des mesures brutes.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, Image

res_dir = Path("/kaggle/working/results_v42")
report = res_dir / "summary_report.md"
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))

for fig_name in [
    "fig1_ablation.png",
    "fig2_noise_robustness.png",
    "fig3_calibration_clinical.png",
    "fig4_mia_privacy_utility.png",
    "fig5_efficiency_all.png",
]:
    p = res_dir / fig_name
    if p.exists():
        print(f"=== {fig_name} ===")
        display(Image(filename=str(p)))

### Synthèse et export
L'archive `/kaggle/working/results_v42.zip` contient :
- `metadata.json` et `summary_report.md` ;
- tous les fichiers CSV bruts et agrégés (`exp1_ablation_raw.csv`, `exp1_ablation_summary.csv`, `exp1_wilcoxon_tests.csv`, `exp2_noise.csv`, `exp3_ood.csv`, `exp4_calibration.csv`, `exp5_regression.csv`, `exp6_iot.csv`, `exp7_mia.csv`, `exp8_efficiency.csv`) ;
- les 5 figures PNG authentiques et le modèle exporté `fanet_v42_pima.onnx`.

Téléchargez `/kaggle/working/results_v42.zip` depuis le panneau Output de Kaggle pour comparer v4.2 face à v4.1.